# M3 X1 on a Kaggle CPU: int8 export, fidelity, package sizes and quantised accuracy

Ruling R100. ONNX file sizes, int8 fidelity and int8 accuracy don't depend on the machine, so they are measured here,
on a Kaggle CPU kernel. `python -m serpentguard.export_matrix matrix` exports R51's matrix with untrained weights at
224 and 320 (and the detectors at 640). Each model is quantised to int8. The transformer encoders, MobileViT-S and
DINOv2, get two variants: static with attention kept fp32, and dynamic. Every int8 file is checked against its fp32 file
on the sample's validation photos and timed. The package is then summed against the 30 MB budget, with an extra total
and fits column for each Q1 retriever (R81). `export_matrix accuracy` then exports each run in `ACCURACY_RUNS` from its
`best.pt` and scores its int8 file(s) with `quantized_accuracy` on the validation photos, against the run's full
validation counts.

**Latency here is the secondary reference**, labelled "Kaggle CPU" in every latency row, in package.csv and in
versions.json. It is never the laptop's. The primary latency, on Puneet's laptop CPU (the O6 device), is measured
there separately. The CPU's model name is read from `/proc/cpuinfo`.

Outputs, in `/kaggle/working/x1/`:
- `latency.csv`, `quantization.csv`, `package.csv`, `calibration_photos.csv` (subset paths only) and `versions.json`;
- `accuracy_<run_id>.json` per accuracy run;
- each command's log, in `logs/`.

ONNX files go to `/kaggle/tmp/x1_onnx`, and each model's are deleted once it is measured. Rerunning a run cell
continues where it stopped.

Code: `../src` and `../results/m3/q1/retrieval.csv` (Q1's retriever sizes). The runner kernel unpacks them into
`/kaggle/working/repo`. Inputs:
- the datasets `serpentguard-indian-snakes-clean` and `serpentguard-m3-subset`, found with
  `kagglehub.dataset_download`;
- the photo sample: the output of `SAMPLE_KERNEL`, i.e. `manifest.csv` with 200 "quant" training photos and 300 "val"
  validation photos, the photos at `data/<path>` beside it;
- each accuracy run: the output of its training kernel.

Both kernel outputs are read through `kagglehub.notebook_output_download`, so declare each of those kernels as a
kernel source.

In [1]:
SAMPLE_KERNEL = "puneetbajajiitm/sg-m3-local-sample"         # "owner/slug" whose output holds manifest.csv and data/
ACCURACY_RUNS = [("puneetbajajiitm/sg-m3-e1a", "E1_mvits")]  # (training kernel, run folder) pairs scored in int8
CPU_MODEL = None     # None reads /proc/cpuinfo's "model name"
SPECS = None         # model names to run (export_matrix --only); None runs the full R51 matrix
SIZES = [224, 320]   # the detectors also run at 640
THREADS = [1, 4]
LATENCY_RUNS = 100   # timed runs per file and thread count, after 10 warm-up runs, then repeated in a fresh session

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; calibration photos go through its eval normalisation
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders
ONNX_VERSION = "1.23.2"            # requirements-m3.txt
ONNXRUNTIME_VERSION = "1.31.1"     # requirements-m3.txt; quantisation, the fp32-op list and latency depend on it
ONNXSCRIPT_VERSION = "0.7.2"       # requirements-m3.txt; F5's export through torch.export
ULTRALYTICS_VERSION = "8.4.202"    # requirements-m3.txt; YOLO26n-seg, YOLO26n and yolov8n-oiv7 export through it
SMP_VERSION = "0.6.0"              # requirements-m3.txt; segmentation-models-pytorch, the U-Net
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}", f"onnx=={ONNX_VERSION}", f"onnxruntime=={ONNXRUNTIME_VERSION}",
                f"onnxscript=={ONNXSCRIPT_VERSION}", f"ultralytics=={ULTRALYTICS_VERSION}",
                f"segmentation-models-pytorch=={SMP_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import onnx
import onnxruntime
import onnxscript
import segmentation_models_pytorch
import timm
import torch
import ultralytics

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
assert onnx.__version__ == ONNX_VERSION, f"onnx {onnx.__version__} is loaded"
assert onnxruntime.__version__ == ONNXRUNTIME_VERSION, f"onnxruntime {onnxruntime.__version__} is loaded"
assert onnxscript.__version__ == ONNXSCRIPT_VERSION, f"onnxscript {onnxscript.__version__} is loaded"
assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
assert segmentation_models_pytorch.__version__ == SMP_VERSION, (f"smp {segmentation_models_pytorch.__version__} "
                                                                f"is loaded")
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}, onnx {onnx.__version__}, "
      f"onnxruntime {onnxruntime.__version__}, onnxscript {onnxscript.__version__}, "
      f"ultralytics {ultralytics.__version__}, smp {segmentation_models_pytorch.__version__}, "
      f"torch {torch.__version__}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
albumentations 2.0.8, timm 1.0.31, onnx 1.23.2, onnxruntime 1.31.1, onnxscript 0.7.2, ultralytics 8.4.202, smp 0.6.0, torch 2.11.0+cpu


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: quantized_accuracy's metrics use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import platform
import time
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import pandas as pd

from serpentguard.export_matrix import (ACCURACY_FILE, LATENCY_FILE, MANIFEST, NEEDS_CONTEXT_EXIT, PACKAGE_FILE,
                                        QUANTIZATION_FILE, VERSIONS_FILE, WITH_RETRIEVER_FITS, WITH_RETRIEVER_MB,
                                        model_specs, read_retrievers, read_sample)
from serpentguard.train import CHECKPOINT_FILE

OWNER = "puneetbajajiitm"
DEVICE = "Kaggle CPU"  # the secondary latency reference; the laptop CPU (the O6 device) is timed separately
OUT = Path("/kaggle/working/x1")
LOGS = OUT / "logs"
ONNX_ROOT = Path("/kaggle/tmp/x1_onnx")  # scratch, outside the saved output; each model's files go once measured
RETRIEVERS = CODE.parent / "results" / "m3" / "q1" / "retrieval.csv"  # Q1's retriever sizes, for R81


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


def kernel_output(handle):
    """A kernel's output; a batch kernel reads it only when that kernel is declared as a kernel source."""
    return Path(kagglehub.notebook_output_download(handle))


def cpu_model():
    """The CPU's model name, as /proc/cpuinfo gives it."""
    cpuinfo = Path("/proc/cpuinfo")
    lines = cpuinfo.read_text().splitlines() if cpuinfo.exists() else []
    names = [line.split(":", 1)[1].strip() for line in lines if line.startswith("model name")]
    return names[0] if names else platform.processor() or None


CPU_MODEL = CPU_MODEL or cpu_model()
DATA = input_root("serpentguard-indian-snakes-clean")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
manifests = sorted(kernel_output(SAMPLE_KERNEL).rglob(MANIFEST))
assert len(manifests) == 1, f"{SAMPLE_KERNEL}'s output must hold one {MANIFEST}; found {manifests}"
SAMPLE = manifests[0].parent
calibration, held_out = read_sample(SAMPLE)  # 200 training photos under train/, validation photos under val/
absent = [file for file in [*calibration["file"], *held_out["file"]] if not Path(file).is_file()]
assert not absent, f"{len(absent)} sample photos are missing, e.g. {absent[0]}"

runs = {}
for kernel, run_id in ACCURACY_RUNS:
    found = sorted(path.parent for path in kernel_output(kernel).rglob(CHECKPOINT_FILE) if path.parent.name == run_id)
    assert len(found) == 1, f"{kernel}'s output must hold one {run_id} folder with {CHECKPOINT_FILE}; found {found}"
    runs[run_id] = found[0]

for required in (DATA / "metadata.csv", DATA / "classes.csv", SUBSET, RETRIEVERS):
    assert required.exists(), f"missing input {required}"
retrievers = read_retrievers(RETRIEVERS)
specs = [spec for spec in model_specs(SIZES) if not SPECS or spec.name in SPECS]
unknown = sorted(set(SPECS or ()) - {spec.name for spec in specs})
assert not unknown, f"SPECS names models outside the matrix: {unknown}"
for folder in (OUT, LOGS, ONNX_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
print(f"CPU: {CPU_MODEL}, {os.cpu_count()} logical cores; latency labelled {DEVICE!r}")
print(f"sample: {SAMPLE}, {len(calibration)} calibration and {len(held_out)} validation photos")
print(f"{len(specs)} model x size specs; accuracy runs:",
      ", ".join(f"{name} ({folder})" for name, folder in runs.items()))
print("retrievers (MB):", {name: round(mb, 2) for name, mb in retrievers.items()})

CPU: Intel(R) Xeon(R) CPU @ 2.20GHz, 4 logical cores; latency labelled 'Kaggle CPU'
sample: /kaggle/input/notebooks/puneetbajajiitm/sg-m3-local-sample/local_sample, 200 calibration and 300 validation photos
28 model x size specs; accuracy runs: E1_mvits (/kaggle/input/notebooks/puneetbajajiitm/sg-m3-e1a/E1_mvits)
retrievers (MB): {'R0': 0.04, 'R1': 0.21, 'R2-L6': 22.94, 'R2-L3': 17.69}


In [5]:
def run_x1(arguments, log_name):
    """python -m serpentguard.export_matrix in its own process, its output in logs/<log_name>.log; returns the exit
    code. Exit 2 means the disk fell under the free-space floor. A rerun skips the models already measured."""
    command = [sys.executable, "-u", "-m", "serpentguard.export_matrix", *arguments, "--sample", str(SAMPLE),
               "--onnx-root", str(ONNX_ROOT), "--out", str(OUT), "--threads", *map(str, THREADS),
               "--runs", str(LATENCY_RUNS), "--latency-device", DEVICE]
    if CPU_MODEL:
        command += ["--cpu-model", CPU_MODEL]
    env = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [str(CODE), os.environ.get("PYTHONPATH")]))}
    started, log_path = time.time(), LOGS / f"{log_name}.log"
    with log_path.open("w") as log:
        code = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, env=env).returncode
    print(f"{log_name}: exit {code} after {(time.time() - started) / 60:.1f} min")
    print(log_path.read_text(errors="replace")[-4000:])
    return code


def check_exit(code, what):
    reason = "the disk fell under the free-space floor" if code == NEEDS_CONTEXT_EXIT else "its log tail is above"
    assert code == 0, f"{what} exited with {code}: {reason}"

In [6]:
matrix = ["matrix", "--sizes", *map(str, SIZES), "--retrievers", str(RETRIEVERS)]
check_exit(run_x1(matrix + (["--only", *SPECS] if SPECS else []), "matrix"), "the matrix")

matrix: exit 1 after 10.5 min
(1, 3, strides=[3, 1], requires_grad=1, device=cpu) = aten::linear(%features, %model.hierarchy_heads.who.weight, %model.hierarchy_heads.who.bias), scope: serpentguard.export.DeployedHeads::/serpentguard.models.snakeid.SnakeID::model/serpentguard.models.heads.HierarchyHeads::hierarchy_heads/torch.nn.modules.linear.Linear::who # /usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:134:0
  return (%17268, %17267, %17271, %17269, %17270)

/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/repo/src/serpentguard/export_matrix.py", line 629, in <module>
    sys.exit(main())
             ~~~~^^
  File "/kaggle/working/repo/src/serpentguard/exp

AssertionError: the matrix exited with 1: its log tail is above

In [7]:
for run_id, folder in runs.items():
    accuracy = ["accuracy", "--run-dir", str(folder), "--data-root", str(DATA), "--subset", str(SUBSET)]
    check_exit(run_x1(accuracy, f"accuracy_{run_id}"), f"accuracy on {run_id}")

accuracy_E1_mvits: exit 0 after 15.7 min
{
  "run_id": "E1_mvits",
  "photos": 300,
  "saved_outputs_top1_agreement": 0.99,
  "static_qdq_attention_fp32": {
    "quantization": "static_qdq_attention_fp32",
    "headline": {
      "fp32": {
        "top1": 0.21666666666666667,
        "top3": 0.43666666666666665,
        "macro_f1": 0.15742917420132607,
        "ece": 0.20555058591334638
      },
      "int8": {
        "top1": 0.04,
        "top3": 0.10333333333333333,
        "macro_f1": 0.024434964941294053,
        "ece": 0.20704483565635848
      }
    },
    "agreement": {
      "top1_agreement": 0.06666666666666667,
      "min_corr": 0.23145754751235892,
      "max_abs_species_logit": 15.285406112670898,
      "mean_embedding_cosine": 0.23253628611564636
    }
  },
  "dynamic": {
    "quantization": "dynamic",
    "headline": {
      "fp32": {
        "top1": 0.21666666666666667,
        "top3": 0.43666666666666665,
        "macro_f1": 0.15742917420132607,
        "ece": 0.205550

In [8]:
versions = json.loads((OUT / VERSIONS_FILE).read_text())
assert versions["machine"]["latency_device"] == DEVICE and not versions["settings"]["stopped_early"]
assert versions["settings"]["models_measured"] == len(specs), versions["settings"]
latency = pd.read_csv(OUT / LATENCY_FILE)
assert set(latency["device"]) == {DEVICE}

quantization = pd.read_csv(OUT / QUANTIZATION_FILE)
print(quantization[["model", "size", "variant", "quantization", "fp32_mb", "int8_mb", "untrained_min_corr",
                    "untrained_top1_agreement", "chosen", "fp32_ops"]].to_string())

package = pd.read_csv(OUT / PACKAGE_FILE)
with_retriever = [column for name in retrievers
                  for column in (WITH_RETRIEVER_MB.format(name), WITH_RETRIEVER_FITS.format(name))]
print(f"\npackage against {versions['settings']['budget_mb']} MB; pipeline_ms timed on {DEVICE}")
print(package[["snakeid", "snakeid_int8", "size", "front_end", "total_mb", "fits", *with_retriever,
               *[column for column in package.columns if column.startswith("pipeline_ms")]]].to_string())

rows = []
for run_id in runs:
    report = json.loads((OUT / ACCURACY_FILE.format(run_id)).read_text())
    for variant, scored in report["variants"].items():
        for precision in ("fp32", "int8"):
            rows.append({"run": run_id, "variant": variant, "precision": precision, **scored["headline"][precision],
                         "int8/fp32 top-1 agreement": scored["agreement"]["top1_agreement"],
                         "int8 MB": report["int8_mb"][variant], "photos": scored["photos"]})
pd.DataFrame(rows).round(4)

AssertionError: 